In [248]:
from src.models.rnn import RNN

In [249]:
import torch

In [250]:
batch_size = 32
sequence_length = 20
input_size = 8

model = RNN(
    input_size=input_size,
    hidden_size=16,
    output_size=1,
)

X = torch.randn(batch_size, sequence_length, input_size)
y_pred = model(X)

print(y_pred.shape)
# torch.Size([32, 1])

torch.Size([32, 1])


In [251]:
from src.data_loader import load_symbol

es_df = load_symbol('ES=F', start='2016-01-01', end='2022-01-01')


In [ ]:
from src.features import build_features




In [255]:
es_features = build_features(es_df)
es_features["y"] = es_features["log_return"].shift(-1)
es_features = es_features.dropna()

y = es_features["y"]
X = es_features.drop(columns=["y"])

n = len(X)
i_val = int(n * 0.6)
i_test = int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i_val], X.iloc[i_val:i_test], X.iloc[i_test:]
y_train, y_val, y_test = y.iloc[:i_val], y.iloc[i_val:i_test], y.iloc[i_test:]

from sklearn.preprocessing import StandardScaler
sx = StandardScaler().fit(X_train)
sy = StandardScaler().fit(y_train.values.reshape(-1, 1))

def make_windows(X, y, L=30):
    Xs, ys = [], []
    for i in range(L, len(X)):
        Xs.append(X[i-L:i])
        ys.append(y[i-1])
    return torch.tensor(np.array(Xs), dtype=torch.float32), torch.tensor(np.array(ys), dtype=torch.float32).unsqueeze(1)

Xtr, ytr = make_windows(sx.transform(X_train), sy.transform(y_train.values.reshape(-1, 1)).ravel())
Xva, yva = make_windows(sx.transform(X_val), sy.transform(y_val.values.reshape(-1, 1)).ravel())
Xte, yte = make_windows(sx.transform(X_test), sy.transform(y_test.values.reshape(-1, 1)).ravel())

In [ ]:
print("Train:", X_train_seq.shape, y_train_seq.shape)
print("Validation:", X_val_seq.shape, y_val_seq.shape)
print("Test:", X_test_seq.shape, y_test_seq.shape)

Train: (1019, 20, 15) (1019, 1)
Validation: (223, 20, 15) (223, 1)
Test: (223, 20, 15) (223, 1)


In [ ]:
train_loader = create_loader(
    X_train_seq, y_train_seq,
    batch_size=32,
    shuffle=True
)

val_loader = create_loader(
    X_val_seq, y_val_seq,
    batch_size=32,
    shuffle=False
)

test_loader = create_loader(
    X_test_seq, y_test_seq,
    batch_size=32,
    shuffle=False
)

In [ ]:
print("Train:", X_train_seq.shape, y_train_seq.shape)
print("Validation:", X_val_seq.shape, y_val_seq.shape)
print("Test:", X_test_seq.shape, y_test_seq.shape)

X_batch, y_batch = next(iter(train_loader))

print("Input batch:", X_batch.shape)
print("Target batch:", y_batch.shape)

Train: (1019, 20, 15) (1019, 1)
Validation: (223, 20, 15) (223, 1)
Test: (223, 20, 15) (223, 1)
Input batch: torch.Size([32, 20, 15])
Target batch: torch.Size([32, 1])


In [ ]:
from src.models.rnn import RNN

model = RNN(
    input_size=X_train_seq.shape[2],
    hidden_size=32,
    output_size=1
)

predictions = model(X_batch)

print("Predictions:", predictions.shape)
print("Targets:", y_batch.shape)

Predictions: torch.Size([32, 1])
Targets: torch.Size([32, 1])


In [ ]:
from torch.optim import Adam
import torch.nn as nn

criterion = nn.MSELoss()

optimizer = Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-5
)






In [ ]:
import copy

epochs = 50

best_val_loss = float('inf')
best_state = None
best_epoch = None

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item() * X_batch.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            total_val_loss += loss.item() * X_batch.size(0)

    val_loss = total_val_loss / len(val_loader.dataset)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train: {train_loss:.8f} | "
        f"Validation: {val_loss:.8f}"
    )

model.load_state_dict(best_state)

print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.8f}")

Epoch 01 | Train: 0.01038919 | Validation: 0.00107502
Epoch 02 | Train: 0.00066589 | Validation: 0.00096728
Epoch 03 | Train: 0.00030513 | Validation: 0.00068286
Epoch 04 | Train: 0.00024088 | Validation: 0.00082963
Epoch 05 | Train: 0.00023136 | Validation: 0.00084310
Epoch 06 | Train: 0.00017987 | Validation: 0.00055612
Epoch 07 | Train: 0.00025645 | Validation: 0.00073390
Epoch 08 | Train: 0.00019399 | Validation: 0.00063815
Epoch 09 | Train: 0.00016760 | Validation: 0.00043218
Epoch 10 | Train: 0.00014646 | Validation: 0.00054299
Epoch 11 | Train: 0.00017722 | Validation: 0.00093319
Epoch 12 | Train: 0.00016733 | Validation: 0.00063428
Epoch 13 | Train: 0.00023814 | Validation: 0.00060550
Epoch 14 | Train: 0.00015582 | Validation: 0.00045283
Epoch 15 | Train: 0.00013646 | Validation: 0.00046212
Epoch 16 | Train: 0.00013477 | Validation: 0.00042872
Epoch 17 | Train: 0.00012723 | Validation: 0.00046817
Epoch 18 | Train: 0.00012104 | Validation: 0.00048729
Epoch 19 | Train: 0.00014404

In [ ]:
model.eval()
total_test_loss = 0.0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        prediction = model(X_batch)
        loss = criterion(y_batch, prediction)
        total_test_loss += loss.item() * X_batch.size(0)
    
test_loss = total_test_loss / len(test_loader.dataset)

test_loss

0.00015062114198779845

In [ ]:
model.eval()
val_predictions = []

with torch.no_grad():
    for X_batch, _ in val_loader:
        pred = model(X_batch)
        val_predictions.extend(pred.squeeze(-1).cpu().numpy())

val_predictions = np.asarray(val_predictions)


In [ ]:
predictions = pd.Series(
    all_predictions,
    index=y_test.index,
    name='prediction'
)

In [ ]:
from src.backtest import Backtest



backtest = Backtest(
    predictions=predictions,
    actual_log_returns=y_test,
    threshold=0.001,
)

results = backtest.order()
strategy_returns = results["strategy_return"]

results.head(100)


,prediction,actual_log_returns,position,strategy_return
Date,,,,
2021-02-12,0.045261,-0.000827,1.0,-0.000827
2021-02-16,0.046314,0.000064,1.0,0.000064
2021-02-17,0.040043,-0.004721,1.0,-0.004710
2021-02-18,0.040999,-0.001664,1.0,-0.001663
2021-02-19,0.042680,-0.007587,1.0,-0.007558
...,...,...,...,...
2021-06-30,0.009117,0.005175,1.0,0.005188
2021-07-01,0.015573,0.007396,1.0,0.007423
2021-07-02,-0.003111,-0.002017,-1.0,0.002019


In [ ]:
print("y_train:")
print(y_train.describe())

print("\ny_test:")
print(y_test.describe())

print("\nPredictions:")
print(predictions.describe())

y_train:
count    1038.000000
mean        0.000330
std         0.011602
min        -0.109552
25%        -0.002527
50%         0.000511
75%         0.004722
max         0.093446
Name: log_return, dtype: float64

y_test:
count    223.000000
mean       0.000857
std        0.008009
min       -0.026346
25%       -0.003365
50%        0.001438
75%        0.005549
max        0.023224
Name: log_return, dtype: float64

Predictions:
count    223.000000
mean       0.020693
std        0.027913
min       -0.035127
25%       -0.001959
50%        0.017119
75%        0.039351
max        0.107298
Name: prediction, dtype: float64


In [ ]:
baseline_train = np.zeros_like(y_train)
baseline_val = np.zeros_like(y_val)

baseline_train_mse = mean_squared_error(baseline_train, y_train)
baseline_val_mse = mean_squared_error(baseline_val, y_val)

print(f"Train baseline MSE: {baseline_train_mse:.8f}")
print(f"Train RNN MSE:      {train_loss:.8f}")

print(f"Val baseline MSE:   {baseline_val_mse:.8f}")
print(f"Val RNN MSE:        {val_loss:.8f}")

Train baseline MSE: 0.00013459
Train RNN MSE:      0.00014056
Val baseline MSE:   0.00020021
Val RNN MSE:        0.00023041


In [ ]:
model.eval()
val_predictions = []

with torch.no_grad():
    for X_batch, _ in val_loader:
        pred = model(X_batch)
        val_predictions.extend(pred.squeeze(-1).cpu().numpy())

val_predictions = np.asarray(val_predictions)

print("y_val mean/std:", y_val.mean(), y_val.std())
print("pred mean/std:", val_predictions.mean(), val_predictions.std())

y_val mean/std: 0.0018399560644633025 0.014060947783464795
pred mean/std: 0.0015092568 0.008255452


In [ ]:
from src.models.gru import GRU

model = GRU(
    input_size=X_train_seq.shape[2],
    hidden_size=32,
    output_size=1
)

predictions = model(X_batch)

print("Predictions:", predictions.shape)
print("Targets:", y_batch.shape)

Predictions: torch.Size([31, 1])
Targets: torch.Size([31, 1])


In [ ]:
import copy

epochs = 50

best_val_loss = float('inf')
best_state = None
best_epoch = None

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item() * X_batch.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    model.eval()
    total_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            total_val_loss += loss.item() * X_batch.size(0)

    val_loss = total_val_loss / len(val_loader.dataset)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train: {train_loss:.8f} | "
        f"Validation: {val_loss:.8f}"
    )

model.load_state_dict(best_state)

print(f"Best epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.8f}")

Epoch 01 | Train: 0.05409786 | Validation: 0.13268270
Epoch 02 | Train: 0.05409786 | Validation: 0.13268270
Epoch 03 | Train: 0.05409786 | Validation: 0.13268270
Epoch 04 | Train: 0.05409786 | Validation: 0.13268270
Epoch 05 | Train: 0.05409786 | Validation: 0.13268270
Epoch 06 | Train: 0.05409786 | Validation: 0.13268270
Epoch 07 | Train: 0.05409786 | Validation: 0.13268270
Epoch 08 | Train: 0.05409786 | Validation: 0.13268270
Epoch 09 | Train: 0.05409786 | Validation: 0.13268270
Epoch 10 | Train: 0.05409786 | Validation: 0.13268270
Epoch 11 | Train: 0.05409786 | Validation: 0.13268270
Epoch 12 | Train: 0.05409786 | Validation: 0.13268270
Epoch 13 | Train: 0.05409786 | Validation: 0.13268270
Epoch 14 | Train: 0.05409786 | Validation: 0.13268270
Epoch 15 | Train: 0.05409786 | Validation: 0.13268270
Epoch 16 | Train: 0.05409786 | Validation: 0.13268270
Epoch 17 | Train: 0.05409786 | Validation: 0.13268270
Epoch 18 | Train: 0.05409786 | Validation: 0.13268270
Epoch 19 | Train: 0.05409786

In [ ]:
model.eval()
total_test_loss = 0.0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        prediction = model(X_batch)
        loss = criterion(y_batch, prediction)
        total_test_loss += loss.item() * X_batch.size(0)
    
test_loss = total_test_loss / len(test_loader.dataset)

test_loss

0.1229953991391199

In [ ]:
model.eval()
val_predictions = []

with torch.no_grad():
    for X_batch, _ in val_loader:
        pred = model(X_batch)
        val_predictions.extend(pred.squeeze(-1).cpu().numpy())

val_predictions = np.asarray(val_predictions)

In [ ]:
predictions = pd.Series(
    all_predictions,
    index=y_test.index,
    name='prediction'
)

In [ ]:
from src.backtest import Backtest



backtest = Backtest(
    predictions=predictions,
    actual_log_returns=y_test,
    threshold=0.001,
)

results = backtest.order()
strategy_returns = results["strategy_return"]

results.head(100)


,prediction,actual_log_returns,position,strategy_return
Date,,,,
2021-02-12,0.045261,-0.000827,1.0,-0.000827
2021-02-16,0.046314,0.000064,1.0,0.000064
2021-02-17,0.040043,-0.004721,1.0,-0.004710
2021-02-18,0.040999,-0.001664,1.0,-0.001663
2021-02-19,0.042680,-0.007587,1.0,-0.007558
...,...,...,...,...
2021-06-30,0.009117,0.005175,1.0,0.005188
2021-07-01,0.015573,0.007396,1.0,0.007423
2021-07-02,-0.003111,-0.002017,-1.0,0.002019


In [ ]:
baseline_train = np.zeros_like(y_train)
baseline_val = np.zeros_like(y_val)

baseline_train_mse = mean_squared_error(baseline_train, y_train)
baseline_val_mse = mean_squared_error(baseline_val, y_val)

print(f"Train baseline MSE: {baseline_train_mse:.8f}")
print(f"Train GRU MSE:      {train_loss:.8f}")

print(f"Val baseline MSE:   {baseline_val_mse:.8f}")
print(f"Val GRU MSE:        {val_loss:.8f}")

Train baseline MSE: 0.00013459
Train GRU MSE:      0.05409786
Val baseline MSE:   0.00020021
Val GRU MSE:        0.13268270


In [ ]:
print(predictions.describe())
print(y_train.describe())
print(y_val.describe())

count    223.000000
mean       0.020693
std        0.027913
min       -0.035127
25%       -0.001959
50%        0.017119
75%        0.039351
max        0.107298
Name: prediction, dtype: float64
count    1038.000000
mean        0.000330
std         0.011602
min        -0.109552
25%        -0.002527
50%         0.000511
75%         0.004722
max         0.093446
Name: log_return, dtype: float64
count    223.000000
mean       0.001840
std        0.014061
min       -0.056743
25%       -0.004356
50%        0.002389
75%        0.010714
max        0.063115
Name: log_return, dtype: float64
